In [2]:
from pathlib import Path
import pandas as pd
import re

# ============================================================
# PATH TO THE MAIN FOLDER CONTAINING ALL XENIUM SAMPLE FOLDERS
# ============================================================

BASE_DIR = Path("/storage2/fs1/sanjayjain/Active/Xenium/Data/KPMP/")


# ============================================================
# SAMPLE LISTS
# ============================================================

MAIN_SAMPLES = [
    "xen37_28-12510",
    "xen37_29-10395",
    "xen37_31-10105",
    "xen37_31-10236",
    "xen38_164-17",
    "xen38_31-10100",
    "xen41_164-21",
    "xen41_29-10010",
    "xen41_29-10404",
    "xen41_935-10542",
    "xen42_27-10154",
    "xen42_28-10051",
    "xen42_936-10948",
    "xen47_31-10000",
    "xen48_27-10042",
    "xen48_28-12263",
    "xen48_29-10277",
    "xen49_164-1",
    "xen50_164-5",
    "xen51_29-10011",
    "xen51_31-10001",
    "xen51_31-10035",
    "xen53_29-11429",
    "xen53_31-10864",
    "xen53_31-11370",
    "xen53_933-11034",
    "xen53_934-10141",
    "xen54_164-7",
    "xen54B_29-11453",
    "xen55_935-10530",
    "xen55_936-10746",
    "xen56_164-3",
    "xen59B_936-10110",
]

EXTERNAL_SAMPLES = [
    "xen31_3781",
    "xen31_3990",
    "xen31_KPMP038",
    "xen32_KPMP101",
]

ALL_SAMPLES = MAIN_SAMPLES + EXTERNAL_SAMPLES


# ============================================================
# FIND THE CORRECT FOLDER FOR EACH SAMPLE
# ============================================================

all_folders = [
    x for x in BASE_DIR.iterdir()
    if x.is_dir()
]


def find_sample_folder(sample):
    """
    Match sample to its folder.

    Preferred:
        xen31_3990

    Also accepts:
        xen31_3990__whatever_else
    """

    # First look for exact folder name
    exact = [
        f for f in all_folders
        if f.name == sample
    ]

    if len(exact) == 1:
        return exact[0]

    # Then allow extra information after sample name.
    # Using a boundary prevents xen49_164-1 from accidentally
    # matching xen49_164-10.
    pattern = re.compile(
        rf"^{re.escape(sample)}(?:__|_|$)"
    )

    matches = [
        f for f in all_folders
        if pattern.match(f.name)
    ]

    if len(matches) == 0:
        print(f"❌ NO FOLDER FOUND: {sample}")
        return None

    if len(matches) > 1:
        print(f"\n⚠️ MULTIPLE FOLDERS FOUND FOR {sample}:")
        for m in matches:
            print("   ", m.name)
        return None

    return matches[0]


# ============================================================
# READ metrics_summary.csv
# ============================================================

results = []

for sample in ALL_SAMPLES:

    folder = find_sample_folder(sample)

    if folder is None:
        results.append({
            "sample": sample,
            "group": (
                "main"
                if sample in MAIN_SAMPLES
                else "external"
            ),
            "status": "folder not uniquely identified"
        })
        continue

    # Usually metrics_summary.csv is directly inside the output folder.
    # rglob also handles it if it happens to be one level deeper.
    metrics_files = list(folder.rglob("metrics_summary.csv"))

    if len(metrics_files) == 0:
        print(f"❌ metrics_summary.csv not found: {sample}")
        results.append({
            "sample": sample,
            "folder": folder.name,
            "status": "metrics_summary.csv not found"
        })
        continue

    if len(metrics_files) > 1:
        print(f"\n⚠️ Multiple metrics_summary.csv files for {sample}:")
        for m in metrics_files:
            print("   ", m)
        continue

    metrics_file = metrics_files[0]

    df = pd.read_csv(metrics_file)

    # Grab any metric containing these terms
    keywords = [
        "negative control",
        "transcripts per cell",
        "transcripts decoded",
        "q20",
    ]

    wanted_cols = [
        col for col in df.columns
        if any(keyword in col.lower() for keyword in keywords)
    ]

    row = {
        "sample": sample,
        "group": (
            "main"
            if sample in MAIN_SAMPLES
            else "external"
        ),
        "folder": folder.name,
        "status": "OK",
    }

    for col in wanted_cols:
        row[col] = df.iloc[0][col]

    results.append(row)


# ============================================================
# COMBINE RESULTS
# ============================================================

qc_summary = pd.DataFrame(results)

display(qc_summary)


# ============================================================
# SAVE
# ============================================================

qc_summary.to_csv(
    BASE_DIR / "Xenium_sample_QC_summary.csv",
    index=False
)

print(
    f"\nSaved to: "
    f"{BASE_DIR / 'Xenium_sample_QC_summary.csv'}"
)

,sample,group,folder,status,fraction_transcripts_decoded_q20,fraction_predesigned_transcripts_decoded_q20,fraction_custom_transcripts_decoded_q20
0,xen37_28-12510,main,xen37_28-12510,OK,0.822964,NaN,NaN
1,xen37_29-10395,main,xen37_29-10395,OK,0.849939,NaN,NaN
2,xen37_31-10105,main,xen37_31-10105,OK,0.921793,NaN,NaN
3,xen37_31-10236,main,xen37_31-10236,OK,0.774847,NaN,NaN
4,xen38_164-17,main,xen38_164-17,OK,0.861416,NaN,NaN
5,xen38_31-10100,main,xen38_31-10100,OK,0.857971,NaN,NaN
6,xen41_164-21,main,xen41_164-21,OK,0.947883,NaN,NaN
7,xen41_29-10010,main,xen41_29-10010,OK,0.860835,NaN,NaN
8,xen41_29-10404,main,xen41_29-10404,OK,0.861817,NaN,NaN
9,xen41_935-10542,main,xen41_935-10542,OK,0.840121,NaN,NaN



Saved to: /storage2/fs1/sanjayjain/Active/Xenium/Data/KPMP/Xenium_sample_QC_summary.csv


In [4]:
# Pick the first sample
sample = MAIN_SAMPLES[0]

# Find its folder using the function from the earlier code
folder = find_sample_folder(sample)

# Find metrics_summary.csv
metrics_file = list(folder.rglob("metrics_summary.csv"))[0]

# Read it
test = pd.read_csv(metrics_file)

# Show all columns related to controls
for col in test.columns:
    if "control" in col.lower():
        print(col)

adjusted_negative_control_probe_rate
adjusted_negative_control_codeword_rate
adjusted_genomic_control_probe_rate
negative_control_probe_counts_per_control_per_cell
genomic_control_probe_counts_per_control_per_cell


In [5]:
results = []

for sample in ALL_SAMPLES:

    folder = find_sample_folder(sample)

    if folder is None:
        continue

    metrics_file = list(folder.rglob("metrics_summary.csv"))[0]
    df = pd.read_csv(metrics_file)

    results.append({
        "sample": sample,
        "neg_control_probe_rate":
            df.loc[0, "adjusted_negative_control_probe_rate"]
    })

neg_probe_rates = pd.DataFrame(results)

neg_probe_rates["neg_control_probe_rate_percent"] = (
    neg_probe_rates["neg_control_probe_rate"] * 100
)

display(neg_probe_rates)

,sample,neg_control_probe_rate,neg_control_probe_rate_percent
0,xen37_28-12510,0.001193,0.119299
1,xen37_29-10395,0.001064,0.106383
2,xen37_31-10105,0.001619,0.161902
3,xen37_31-10236,0.000884,0.088439
4,xen38_164-17,0.001116,0.111635
5,xen38_31-10100,0.001179,0.117917
6,xen41_164-21,0.001139,0.113866
7,xen41_29-10010,0.000650,0.064969
8,xen41_29-10404,0.000653,0.065298
9,xen41_935-10542,0.000505,0.050539
